# Group 1 — Project 3 (breathing & heart-rate variability)

A physiological-sigh / breath-hold protocol (inspired by Andrew Huberman): establish a baseline, hold an exhale until you can't, let the heart rate recover on its own, hold again, then try to *actively* bring it back down. This notebook loads the EKG, finds the heartbeats, and tracks how the **inter-beat interval** moves.

**Sensors** — Delsys `Ch12` = EKG (chest), `Ch1` = right-trapezius EMG. (Single modality — no cross-device sync here.)

In [ ]:
!pip install -q delsys pysampled huggingface_hub

### 1 · Get your data

In [ ]:
from huggingface_hub import hf_hub_download
REPO = "praneethnamburimit/immersionlab-pe-mis-groups"
csv = hf_hub_download(REPO, "group_1_3/delsys/Trial_1.csv", repo_type="dataset")
print(csv)

### 2 · Load it + look at the EKG

In [ ]:
import delsys, numpy as np, matplotlib.pyplot as plt
lf = delsys.Log(csv)
ekg = lf.find(sensor_number=12, as_="sensor")[0].ekg
plt.figure(figsize=(12,3)); plt.plot(ekg.t, ekg()); plt.xlim(30,40); plt.title("EKG (30–40 s)"); plt.xlabel("s"); plt.show()

### 3 · Heartbeats → inter-beat interval over the whole run

In [ ]:
peaks = np.asarray(ekg.find_rpeaks()); tp = ekg.t[peaks]
ibi = np.diff(tp)*1000.0
plt.figure(figsize=(13,3)); plt.plot(tp[1:], ibi, "-o", ms=2)
plt.ylabel("IBI (ms)"); plt.xlabel("time (s)"); plt.title(f"{len(peaks)} beats — watch the IBI stretch and shrink with the breath holds"); plt.show()
print(f"mean HR ~ {60000/np.median(ibi):.0f} bpm")

### 4 · Your turn
- Mark your **baseline / breath-hold / recovery / active-recovery** windows (you know the order) and compare **RMSSD** (`np.sqrt(np.mean(np.diff(ibi)**2))`) across them — does the active recovery pull it back faster?
- The **trapezius EMG** is on `sensor_number=1` (`.emg.bandpass(20,450).envelope()`) — does neck/shoulder tension rise during the holds?
> The detector will miss/add a beat here and there; clean the obvious outliers before trusting RMSSD.